**Notebook Feito por:** MSc. Eng. Paulo de Souza Silva  
**Data:** Agosto de 2026  
**Conteúdo retirado e adaptado de livros e artigos sobre Galerkin Descontínuo**  
**Agradecimentos:** Um agradecimento ao Prof. Dr. Alberto Nogueira pela disponibilização dos scripts em Python para DG 1D

# **Aula 08 - Fluxos Numéricos**

## **Introdução**

Na aula anterior, desvendamos o funcionamento do **termo de volume** da formulação do Método de Galerkin Descontínuo (DG). Vimos como realizar a **projeção dos fluxos físicos** no espaço polinomial e como essa projeção, em conjunto com a Matriz de Rigidez ($\mathcal{S}$), descreve corretamente toda a dinâmica que acontece **no interior de cada elemento** da malha.

Entretanto, ainda existe uma peça importante que falta para completarmos o método.

Para enxergarmos isso com clareza, vamos retomar a formulação fraca (semi-discreta) da equação que estamos resolvendo:

$$
\underbrace{J_e \left[\int_{-1}^{1} \phi_i \phi_j d \xi \right]}_{\text{Jacobiano e Matriz de Massa} \ \mathcal{M}}
\dfrac{\partial}{\partial t} \begin{Bmatrix} c_0 \\ c_1 \\ \vdots \\ c_P \end{Bmatrix} =
\underbrace{\int_{-1}^{1} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi }_{\text{Fluxo, Matriz de Derivação} \ \mathcal{D} \ \text{e Rigidez} \ \mathcal{S}} -
\underbrace{\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} +
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}}_{\text{Fluxos nas Fronteiras}}
$$

Observe cada um dos blocos da equação.

O lado esquerdo controla a evolução temporal da solução por meio da Matriz de Massa ($\mathcal{M}$). O primeiro termo do lado direito representa toda a dinâmica que ocorre **dentro do elemento**, sendo justamente o operador que estudamos na aula anterior.

Mas e os dois últimos termos destacados?

É exatamente aqui que está um dos conceitos mais importantes do Método de Galerkin Descontínuo.

Como o próprio nome do método sugere, os elementos da malha são **descontínuos**. Em outras palavras, dois elementos vizinhos **não compartilham necessariamente o mesmo valor na interface**. Cada elemento possui sua própria aproximação polinomial e, consequentemente, podem existir dois valores diferentes exatamente na mesma posição física da fronteira.

Isso nos leva naturalmente à seguinte pergunta:

> **Se existem dois valores diferentes na interface, qual deles deve ser utilizado para calcular o fluxo que atravessa essa fronteira?**

Essa pergunta pode parecer simples, mas ela está no coração dos métodos conservativos para Equações Diferenciais Parciais. A resposta não é imediata e, durante décadas, motivou o desenvolvimento de diversas estratégias numéricas capazes de determinar corretamente o fluxo entre duas regiões vizinhas.

No Método de Galerkin Descontínuo, essas estratégias aparecem através dos termos

$$\tilde{f}_{e,e+1} \qquad\text{e}\qquad \tilde{f}_{e-1,e},$$

conhecidos como **Fluxos Numéricos**.

Podemos interpretar esses fluxos como um **árbitro matemático**. Eles observam os dois estados existentes na interface (um vindo do elemento à esquerda e outro do elemento à direita) e determinam um **único fluxo** que será utilizado por ambos os elementos.

Em outras palavras, os Fluxos Numéricos representam o **último componente necessário** para construirmos completamente o operador espacial do Método de Galerkin Descontínuo, denotado por $L_h,$ que posteriormente será integrado no tempo por métodos como Runge-Kutta.

### **Por que estudar Fluxos Numéricos?**
Embora os Fluxos Numéricos sejam fundamentais no DG, eles **não surgiram originalmente nesse método**. 

Na realidade, a necessidade de definir corretamente o fluxo entre regiões vizinhas apareceu muitos anos antes, principalmente em métodos de **Diferenças Finitas** e **Volumes Finitos**, onde também era necessário determinar como a informação deveria atravessar as interfaces entre células.

Por esse motivo, antes de estudarmos os Fluxos Numéricos dentro do contexto do DG, faremos uma breve viagem até esses métodos clássicos. Essa abordagem nos permitirá compreender **por que os Fluxos Numéricos surgiram**, quais problemas eles resolvem e qual é a motivação física por trás dos chamados **Solvers de Riemann**.

Depois dessa motivação, voltaremos naturalmente ao Método de Galerkin Descontínuo e veremos como essas mesmas ideias são incorporadas na formulação do DG.

---

## **Esquemas Numéricos em Diferenças Finitas**

Os esquemas numéricos surgem com naturalidade ao se avaliar as EDPs de forma discretizada via Diferenças Finitas. O exemplo mais clássico para compreender essa origem é a equação de convecção linear unidimensional $f(u) = au$. 

Apesar de extremamente simples, ela possui todas as características fundamentais de problemas hiperbólicos: a informação propaga-se na forma de ondas ao longo do domínio. Isso faz dela um excelente laboratório para entendermos como diferentes discretizações influenciam a estabilidade e a precisão da solução.

Considere a equação de convecção linear 1D, que descreve a propagação de uma onda ao longo do eixo-x com velocidade $a$
$$\frac{\partial u}{\partial t}+a\frac{\partial u}{\partial x}=0$$

Se $a$ for positivo, a solução de onda progressiva da equação acima propaga-se para a direita; o lado esquerdo é denominado lado *upwind* (a montante do fluxo) e o lado direito é o lado *downwind* (a jusante do fluxo). De modo análogo, se $a$ for negativo, a solução de onda progressiva propaga-se para a esquerda; o lado esquerdo é denominado lado *downwind* e o lado direito é o lado *upwind*. 

Se a aproximação da derivada espacial for construída utilizando preferencialmente informações provenientes do lado *upwind* (de onde a informação física está chegando), o esquema é denominado **esquema Upwind**.


### **Esquema Upwind de Primeira Ordem** 

No curso da professora Lorena Barba (ver aula 02_01 1D Convection) um problema desse tipo pode ser escrito de forma discretizada com a estratégia *forward* no tempo e *backward* no espaço quando $a > 0$ 

$$\frac{u_i^{n+1}-u_i^n}{\Delta t} + a \frac{u_i^n - u_{i-1}^n}{\Delta x} = 0$$

e caso $a < 0 $ devemos adotar *forward* no espaço também
$$\frac{u_i^{n+1}-u_i^n}{\Delta t} + a \frac{u_{i+1}^n - u_{i}^n}{\Delta x} = 0$$

Vamos considerar que $a$ é positivo e reorganiza-lo, o que nos leva:
$$\boxed{u_i^{n+1} = u_i^n - a \frac{\Delta t}{\Delta x}(u_i^n-u_{i-1}^n)}$$

Essa técnica é estável se respeitar o coecifiente de Courant-Friedrichs-Lewy (CFL): 

$$CFL = \left|\frac{a \Delta t}{\Delta x} \right| \leq 1$$

Em palavras, a informação não pode percorrer mais do que uma célula durante um único passo de tempo. Quando essa condição é violada, o método deixa de representar corretamente a propagação da informação física e a solução torna-se instável.  

#### **Exemplo de Aplicação**
> Referência: https://www.psvolpiani.com/courses

Vamos considerar o problema de advecção linear descrito de forma discretizada por:
$$u_i^{n+1} = u_i^n - a \frac{\Delta t}{\Delta x}(u_i^n-u_{i-1}^n)$$

Para tal, vamos assumir como condição Inicial:
$$u_0(x) = \exp(-0.5(x/0.4)^2)$$

Condição de contorno:
$$u(-2,t) = u(2,t)$$

Para esse exemplo tomaremos $a = 0.8$ e $\Delta x = 0.04$. Avaliaremos a mudança do valor de $\Delta t$ para mudar o CFL:

<div align="center">

| Parâmetros | Caso 1 | Caso 2 | Caso 3 |
|:----------:|:------:|:------:|:------:|
|$a$         |0.8     |0.8     |0.8     |
|$\Delta x$  |0.04    |0.04    |0.04    |
|$\Delta t$  |0.04    |0.05    |0.06    |
|$CFL$       | 0.8    |1.0     | 1.2    |

</div>



In [1]:
from IPython.display import HTML, display

# display(HTML("""
# <div style="display:flex; gap:10px;">
#     <img src="animations/advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
#     <img src="animations/advection_Nx101_CFL1.00_T5.0/simulation.gif" width="400">
#     <img src="animations/advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
# </div>
# """))

#### **O Comportamento da Solução e o Limite CFL**

Como podemos observar nas animações acima, o comportamento do esquema *Upwind* de primeira ordem é extremamente sensível à escolha do passo de tempo $\Delta t$ e do espaçamento da malha $\Delta x$. Analisando os três cenários:

1. **$c = 0.8$ (Dissipação Numérica):** A simulação permanece estável, porém a onda perde amplitude e torna-se progressivamente mais "achatada". Embora o critério CFL seja satisfeito, o erro de truncamento do esquema introduz uma difusão (ou viscosidade) artificial, fazendo com que a solução seja suavizada ao longo do tempo.

2. **$c = 1.0$ (Cenário Ideal):** A onda desloca-se perfeitamente para a direita, preservando sua forma e amplitude. Nesse caso, a informação percorre exatamente um nó da malha a cada passo de tempo, de modo que cada valor é simplesmente transferido para o nó vizinho, sem introduzir dissipação numérica.

3. **$c = 1.2$ (Instabilidade):** A solução diverge rapidamente. Do ponto de vista numérico, a informação física propaga-se mais rapidamente do que a malha consegue transmiti-la. Como consequência, o domínio de dependência numérico deixa de conter o domínio de dependência físico, violando o critério de estabilidade CFL.

Embora o esquema *Upwind* consiga ser estável quando o critério CFL é respeitado, sua elevada dissipação numérica reduz significativamente a precisão da solução. Surge então uma pergunta natural:

> **É possível reduzir essa dissipação sem comprometer a estabilidade do método?**

Uma primeira ideia consiste em abandonar a aproximação unilateral da derivada espacial e utilizar uma diferença central, que possui maior ordem de precisão:

$$\frac{u_i^{n+1}-u_i^n}{\Delta t}+a\frac{u_{i+1}^n-u_{i-1}^n}{2\Delta x}=0$$

Esse é o conhecido esquema **FTCS** (*Forward in Time, Central in Space*). 

À primeira vista, ele parece uma escolha mais precisa do que o esquema *Upwind*. Entretanto, a análise de von Neumann demonstra que, para problemas puramente convectivos, o FTCS é **incondicionalmente instável**, independentemente do valor de $\Delta t$.

Diante dessa limitação, diversos pesquisadores passaram a desenvolver novas discretizações capazes de combinar estabilidade e precisão.

---

### **Esquema de Lax-Friedrichs**

Entre as primeiras propostas destaca-se o Esquema de Lax-Friedrichs que modifica o FTCS substituindo o termo $u_i^n$ da derivada temporal pela média dos estados vizinhos, 

$$u_i^n \;\longrightarrow\; \frac{1}{2}(u_{i-1}^n+u_{i+1}^n)$$

substituindo na FTCS
$$\frac{u_i^{n+1}-\frac{1}{2}(u_{i-1}^n+u_{i+1}^n)}{\Delta t}+a\frac{u_{i+1}^n-u_{i-1}^n}{2\Delta x}=0$$

reescrevendo para isolar $u_i^{n+1}$

$$\boxed{u_i^{n+1}= \frac{1}{2}(u_{i-1}^n+u_{i+1}^n) - a\frac{\Delta t}{2\Delta x} (u_{i+1}^n-u_{i-1}^n)}$$


In [4]:
# display(HTML("""
# <div style="display:flex; gap:10px;">
#     <img src="animations/advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
#     <img src="animations/LFF_advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
# </div>
# """))

Comparando os dois resultados, percebe-se que o esquema de Lax-Friedrichs apresenta uma dissipação numérica ainda maior que o esquema Upwind. A maior estabilidade é obtida às custas de uma suavização mais intensa da solução, fazendo com que o pico da onda perca amplitude mais rapidamente.

In [3]:
# display(HTML("""
# <div style="display:flex; gap:10px;">
#     <img src="animations/advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
#     <img src="animations/LFF_advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
# </div>
# """))

Embora o esquema de Lax-Friedrichs possua uma dissipação numérica significativamente maior que o Upwind, ambos permanecem sujeitos ao critério de estabilidade CFL. Quando $CFL>1$, a dissipação adicional pode retardar visualmente o crescimento das oscilações, mas não é suficiente para impedir que a solução se torne instável.

> Fica como sugestão de leitura sobre o Esquema de Lax-Wendroff

---

## **Fluxos Numéricos Convectivos**

### **Forma Discreta Conservativa de Volumes Finitos**

Em Volumes Finitos (VF) a estratégia de **discretização** da malha é diferente do método das Diferenças Finitas. Vamos considerar o problema da lei de conservação escalar unidimensional:
$$\frac{\partial u}{\partial t} + \frac{\partial f(u)}{\partial x} = 0$$

Considere um domínio espacial discretizado em células (ou volumes de controle) de tamanho $\Delta x$. A célula $i$ está compreendida no intervalo $[x_{i-1/2}, x_{i+1/2}]$. O tempo é avançado em passos discretos $\Delta t$, do instante $t^n$ para $t^{n+1}$.


A ideia central do MVF é integrar a equação governante sobre o volume de controle espacial e o intervalo de tempo de interesse. Vamos aplicar a integral dupla:
$$\int_{t^n}^{t^{n+1}} \int_{x_{i-1/2}}^{x_{i+1/2}} \left( \frac{\partial u}{\partial t} + \frac{\partial f(u)}{\partial x} \right) dx dt = 0$$

Devido à linearidade da integral, podemos separar a equação em dois termos (um temporal e um espacial) e trocar a ordem de integração onde for matematicamente conveniente.

Para o termo com a derivada no tempo, integramos primeiro em $t$:

$$\int_{x_{i-1/2}}^{x_{i+1/2}} \left[ \int_{t^n}^{t^{n+1}} \frac{\partial u}{\partial t} dt \right] dx = \int_{x_{i-1/2}}^{x_{i+1/2}} \left[ u(x, t^{n+1}) - u(x, t^n) \right] dx$$

No Método dos Volumes Finitos, definimos o valor médio da célula no instante $t$ como:
$$u_i(t) = \frac{1}{\Delta x} \int_{x_{i-1/2}}^{x_{i+1/2}} u(x, t) dx$$

Substituindo essa definição, a integral do termo temporal se torna simplesmente:

$$\Delta x \left( u_i^{n+1} - u_i^n \right)$$

Para o termo da derivada espacial, integramos primeiro em $x$. Usando o Teorema Fundamental do Cálculo na integral interna:
$$\int_{t^n}^{t^{n+1}} \left[ \int_{x_{i-1/2}}^{x_{i+1/2}} \frac{\partial f(u)}{\partial x} dx \right] dt = \int_{t^n}^{t^{n+1}} \left[ f(u(x_{i+1/2}, t)) - f(u(x_{i-1/2}, t)) \right] dt$$

Definimos o fluxo numérico médio através da interface $x_{i+1/2}$ durante o intervalo de tempo $\Delta t$ como:

$$F_{i+1/2} = \frac{1}{\Delta t} \int_{t^n}^{t^{n+1}} f(u(x_{i+1/2}, t)) dt$$

Assim, a integral do termo espacial resulta na diferença dos fluxos na fronteira multiplicada pelo passo de tempo:

$$\Delta t \left( F_{i+1/2} - F_{i-1/2} \right)$$

Somando os dois resultados obtidos e igualando a zero (conforme a equação original da integral dupla), temos o balanço exato:

$$\Delta x \left( u_i^{n+1} - u_i^n \right) + \Delta t \left( F_{i+1/2} - F_{i-1/2} \right) = 0$$

Agora, basta isolar o termo $u_i^{n+1}$ (o valor médio no novo passo de tempo):

$$\Delta x \left( u_i^{n+1} - u_i^n \right) = -\Delta t \left( F_{i+1/2} - F_{i-1/2} \right)$$

Dividindo toda a equação por $\Delta x$, chegamos à clássica **forma discreta conservativa**:

$$\boxed{u_i^{n+1} = u_i^n - \frac{\Delta t}{\Delta x} \left(F_{i+\frac{1}{2}}-F_{i-\frac{1}{2}}\right)}$$

onde:

- $F_{i+\frac{1}{2}}$ é o **fluxo numérico** calculado na interface direita da célula \(i\);
- $F_{i-\frac{1}{2}}$ é o **fluxo numérico** calculado na interface esquerda da célula \(i\).

Observe que a evolução da solução dentro da célula depende exclusivamente dos fluxos que cruzam suas fronteiras. É exatamente neste ponto que nasce a necessidade fundamental do Fluxo Numérico: como as células vizinhas possuem médias diferentes e geram uma descontinuidade natural na interface, o fluxo físico exato $f(u)$ é desconhecido ali. Ele precisa ser substituído por uma aproximação matemática baseada nos estados da esquerda e da direita.

O fluxo que deixa a célula $i$ pela interface direita é matemática e fisicamente idêntico ao fluxo que entra na célula $i+1$ pela interface esquerda. Dessa forma, ao somarmos todas as células da malha, os fluxos internos se cancelam perfeitamente. Nenhuma gota de fluido ou joule de energia é perdida ou criada numericamente, garantindo que a discretização preserve a física exata da conservação.

---

### **Fluxo Numérico Upwind**

Vamos pegar a equação do Upwind clássico para o caso onde a velocidade é positiva ($a > 0$):

$$u_i^{n+1} = u_i^n - a \frac{\Delta t}{\Delta x}(u_i^n - u_{i-1}^n)$$

Podemos manipular essa equação colocando a velocidade $a$ (que compõe o fluxo físico $f(u)=au$) para dentro dos parênteses:

$$u_i^{n+1} = u_i^n - \frac{\Delta t}{\Delta x}(a u_i^n - a u_{i-1}^n)$$

Comparando diretamente essa equação com a forma conservativa universal apresentada acima,
$$u_i^{n+1} = u_i^n - \frac{\Delta t}{\Delta x} \left(F_{i+\frac{1}{2}}-F_{i-\frac{1}{2}}\right)$$

a correspondência aparace de forma trivial:

$$F_{i+1/2} = a u_i^n \hspace{2cm} F_{i-1/2} = a u_{i-1}^n$$

Acabamos de deduzir o **Fluxo Numérico Upwind**! Matematicamente, ele nos diz que, se a onda viaja para a direita, o fluxo na interface da direita ($i+1/2$) é ditado unicamente pelo estado interno da própria célula $i$.

---

### **Fluxo Numérico de Godunov**

É comum na literatura que o fluxo Upwind seja muitas vezes chamado de Fluxo de Godunov, visto que a estrutura matemática de atualização de ambas as estratégias baseia-se na mesma forma conservativa:

$$q_i^{n+1} = q_i^n - \frac{\Delta t}{\Delta x} \left( f^n_{i+1/2} - f^n_{i-1/2} \right)$$

(Onde $\Delta x = x_{i+1/2} - x_{i-1/2}$ representa o tamanho da célula e $q$ é a nossa variável conservada).

No entanto, a verdade conceitual é que o fluxo Upwind clássico é apenas um caso particular do método de Godunov aplicado a equações lineares.

**Problemas Não-Lineares**

O truque algébrico que deduz o Upwind funciona perfeitamente para equações lineares onde a velocidade de propagação $a$ é constante. Mas e se estivermos resolvendo um sistema não-linear (como as Equações de Euler ou a Equação de Burgers), onde a velocidade da onda muda dependendo da solução local? Em uma **mesma interface, podemos ter ondas viajando para a esquerda e para a direita simultaneamente**.

Godunov propôs que, em vez de usarmos "truques algébricos", deveríamos olhar diretamente para a física local da interface.

Ele assumiu que em cada fronteira $i+1/2$ existe uma descontinuidade abrupta de estados (o chamado Problema de Riemann). O método de Godunov consiste em:
1. Resolver o Problema de Riemann de forma **exata** na interface.
2. Analisar as ondas físicas geradas por essa colisão (choques ou expansões).
3. Extrair o valor exato da solução na linha da interface ao longo do tempo para compor o fluxo numérico.

Em suma, **o Fluxo de Godunov é a generalização física e não-linear do esquema Upwind**. 

---


### **Fluxo Numérico de Roe**

Como debatido, o método de Godunov resolve o Problema de Riemann de forma exata. O grande problema é que, para sistemas de equações não-lineares, como as Equações de Euler (que regem a dinâmica dos fluidos compressíveis), resolver esse problema exato iterativamente em cada interface da malha torna o custo computacional proibitivo.

Para um sistema de equações, a lei de conservação assume a forma vetorial:
$$\frac{\partial \mathbf{U}}{\partial t} + \frac{\partial \mathbf{F}(\mathbf{U})}{\partial x} = 0$$

Onde $\mathbf{U}$ é o vetor de variáveis conservadas (massa, quantidade de movimento, energia) e $\mathbf{F}(\mathbf{U})$ é o vetor de fluxos. Se aplicarmos a regra da cadeia na derivada do fluxo, surge a Matriz Jacobiana do sistema ($\mathbf{A}$):

$$\frac{\partial \mathbf{U}}{\partial t} + \mathbf{A}(\mathbf{U}) \frac{\partial \mathbf{U}}{\partial x} = 0 \qquad \text{onde} \qquad \mathbf{A} = \frac{\partial \mathbf{F}}{\partial \mathbf{U}}$$

O grande desafio computacional é que essa matriz $\mathbf{A}$ varia ponto a ponto. Na interface entre duas células, temos um estado à esquerda ($\mathbf{U}_L$) e um à direita ($\mathbf{U}_R$). Qual Jacobiana devemos usar?

Para evitar o custo computacional do método de Godunov exato, Roe propôs criar um **Solucionador de Riemann Aproximado**. A ideia foi substituir o problema não-linear complexo por um problema linearizado apenas na interface.

Roe provou que é possível construir uma matriz constante especial, chamada de Matriz de Roe $\tilde{\mathbf{A}}(\mathbf{U}_L, \mathbf{U}_R)$, que satisfaz exatamente a seguinte condição (Condição de Roe):
$$\mathbf{F}(\mathbf{U}_R) - \mathbf{F}(\mathbf{U}_L) = \tilde{\mathbf{A}} (\mathbf{U}_R - \mathbf{U}_L)$$

Ao fazer isso, o intrincado problema de descobrir para qual lado cada onda vai se resume a calcular os autovalores e autovetores dessa matriz $\tilde{\mathbf{A}}$. O fluxo numérico de Roe assume então a sua forma clássica:

$$\mathbf{F}_{i+1/2}^{Roe} = \frac{1}{2} \left( \mathbf{F}_L + \mathbf{F}_R \right) - \frac{1}{2} \vert{}\tilde{\mathbf{A}}\vert{} (\mathbf{U}_R - \mathbf{U}_L)$$

Onde o termo $\vert{}\tilde{\mathbf{A}}\vert{}$ representa a matriz avaliada com o valor absoluto de seus autovalores.

O Fluxo de Roe é o Upwind perfeito para sistemas de equações. Ele consegue capturar ondas de choque e descontinuidades de contato de forma extremamente nítida. Entretanto, ele traz dois problemas que abriram portas para outras metodologias:
1. **Custo e Complexidade:** Calcular a matriz $\tilde{\mathbf{A}}$, seus autovalores e autovetores para cada interface é matematicamente trabalhoso (exigem as médias ponderadas de Roe).
2. **Violação de Entropia:** Por ser uma linearização matemática, em regiões onde o fluido passa da velocidade subsônica para supersônica (expansões fortes), o método de Roe pode se "confundir" e criar choques de expansão - um fenômeno fisicamente impossível que viola a Segunda Lei da Termodinâmica.

> Para mais informações ler o **Capítulo 11 de Toro**

Diante dessas dificuldades podemos adotar um método mais simples, que não exige o cálculo de autovetores que são os fluxos de Lax-Friedrichs (e sua versão local, o método de Rusanov).

---

### **Fluxo Numérico de Lax-Friedrichs**

Anteriormente pelo Método das Diferenças Finitas analisando o problema de advecção linear, chegamos no esquema de Lax-Friedrichs escrito como:
$$u_i^{n+1}= \frac{1}{2}(u_{i-1}^n+u_{i+1}^n) - a\frac{\Delta t}{2\Delta x} (u_{i+1}^n-u_{i-1}^n)$$

em um caso mais geral mas ainda para o DF temos que o esquema assume a forma:
$$u_i^{n+1}= \frac{1}{2}(u_{i-1}^n+u_{i+1}^n) - a\frac{\Delta t}{2\Delta x} (f(u_{i+1}^n)-f(u_{i-1}^n))$$

*A partir daqui, vou omitir o sobrescrito $n$ de $u^n$ para deixar a notação mais limpa, assumindo que tudo do lado direito está no tempo $n$.*

Novamente nosso objetivo é transformar essa escrita no formato conservativo de Volumes:
$$u_i^{n+1} = u_i^n - \frac{\Delta t}{\Delta x} \left(F_{i+\frac{1}{2}}-F_{i-\frac{1}{2}}\right)$$

Vamos olhar apenas para o primeiro termo da fórmula original: $\frac{1}{2}(u_{i+1} + u_{i-1})$. Para forçar o $u_i$ a aparecer, nós somamos e subtraímos $u_i$ dentro da equação:

$$\frac{1}{2}(u_{i+1} + u_{i-1}) = u_i - u_i + \frac{1}{2}u_{i+1} + \frac{1}{2}u_{i-1}$$

Agora, agrupamos os termos dividindo o $-u_i$ em duas metades ($-\frac{1}{2}u_i$ e $-\frac{1}{2}u_i$):

$$= u_i + \left( \frac{1}{2}u_{i+1} - \frac{1}{2}u_i \right) - \left( \frac{1}{2}u_i - \frac{1}{2}u_{i-1} \right)$$

Pronto! Já temos o $u_i$ isolado e dois blocos de diferença, um olhando para a direita ($i+1$) e outro para a esquerda ($i-1$)

Agora olhamos para a parte física da equação original: $-\frac{\Delta t}{2 \Delta x} (f(u_{i+1}) - f(u_{i-1}))$.

Vamos usar a mesma lógica, somando e subtraindo $f(u_i)$ lá dentro para separar as fronteiras:
$$f(u_{i+1}) - f(u_{i-1}) = f(u_{i+1}) \mathbf{+ f(u_i) - f(u_i)} - f(u_{i-1})$$

Rearranjando para criar agrupamentos da direita e da esquerda:
$$= \left( f(u_{i+1}) + f(u_i) \right) - \left( f(u_i) + f(u_{i-1}) \right)$$

Multiplicando tudo pela constante que estava na frente, obtemos:
$$- \frac{\Delta t}{\Delta x} \left[ \frac{f(u_{i+1}) + f(u_i)}{2} - \frac{f(u_i) + f(u_{i-1})}{2} \right]$$

Agora, vamos substituir o que encontramos no Passo 3 e no Passo 4 de volta na equação principal:
$$u_i^{n+1} = u_i + \left[ \frac{1}{2}(u_{i+1} - u_i) - \frac{1}{2}(u_i - u_{i-1}) \right] - \frac{\Delta t}{\Delta x} \left[ \frac{f(u_{i+1}) + f(u_i)}{2} - \frac{f(u_i) + f(u_{i-1})}{2} \right]$$

Para colocar no formato conservativo estrito, precisamos colocar o termo $-\frac{\Delta t}{\Delta x}$ em evidência total. 

Isso significa que precisamos multiplicar o colchete das variáveis $u$ pelo inverso $\frac{\Delta x}{\Delta t}$. 

Fica assim:
$$u_i^{n+1} = u_i - \frac{\Delta t}{\Delta x} \Bigg[ \left( \frac{f(u_{i+1}) + f(u_i)}{2} - \frac{\Delta x}{2 \Delta t}(u_{i+1} - u_i) \right) - \left( \frac{f(u_i) + f(u_{i-1})}{2} - \frac{\Delta x}{2 \Delta t}(u_i - u_{i-1}) \right) \Bigg]$$

O que está no primeiro parênteses (que usa os índices $i+1$ e $i$) é exatamente o seu fluxo na fronteira direita:
$${F_{i+1/2} = \frac{1}{2}(f(u_i) + f(u_{i+1})) - \frac{\Delta x}{2 \Delta t}(u_{i+1} - u_i)}$$

O que está no segundo parênteses (usando $i$ e $i-1$) é exatamente o fluxo na fronteira esquerda:
$${F_{i-1/2} = \frac{1}{2}(f(u_{i-1}) + f(u_i)) - \frac{\Delta x}{2 \Delta t}(u_i - u_{i-1})}$$

> Quando formos programar isso no código DG, faremos apenas duas adaptações puramente conceituais à fórmula do $F_{i+1/2}$:
> 1. Nomenclatura: Em vez de usar $i$ e $i+1$, você chamará de estado interno ($u_{esq}$) e estado externo ($u_{dir}$).
> 2. Viscosidade Numérica: O termo $\frac{\Delta x}{\Delta t}$ vem direto da malha. No DG e Volumes Finitos, é de costume trocar esse fator geométrico pela **maior velocidade de onda local**, geralmente chamada de constante $C = \max \vert{}f'(u)\vert{}$. Isso torna o método muito mais flexível.

Assim, o **Fluxo Numérico de Lax-Friedrich** para o DG é:
$$\boxed{F_{LxF}^{DG} = \frac{1}{2}(f(u_{esq}) + f(u_{dir})) - \frac{C}{2}(u_{dir} - u_{esq})}$$

---


### **Fluxo Numérico de Rusanov (LxF Local)**

O fluxo de Lax-Friedrichs que acabamos de deduzir é extremamente sofisticado, mas esconde um problema prático. Se utilizarmos a maior velocidade de onda de todo o domínio para definir a constante $C$ (o que chamamos de Lax-Friedrichs Global), estaremos injetando a mesma quantidade massiva de dissipação numérica em todas as interfaces da malha - inclusive naquelas onde o escoamento está calmo. Isso introduz muita viscosidade artificial e "borra" excessivamente a solução.

Para contornar isso, Victor Rusanov (1961) propôs uma modificação simples: tornar a dissipação **local**.

Em vez de buscar a velocidade máxima de todo o problema, o método de Rusanov (frequentemente chamado de Lax-Friedrichs Local, ou LLxF) avalia a maior velocidade de onda apenas entre os dois estados que compartilham aquela interface específica:

$$C_{local} = \max(\vert{}f'(u_{esq})\vert{}, \vert{}f'(u_{dir})\vert{})$$

Assim, o Fluxo Numérico de Rusanov é escrito como:
$$\boxed{F_{Rusanov} = \frac{1}{2}(f(u_{esq}) + f(u_{dir})) - \frac{C_{local}}{2}(u_{dir} - u_{esq})}$$

O fluxo de Rusanov preserva a simplicidade extrema do Lax-Friedrichs e garante que a física não seja violada (não gera choques de expansão impossíveis, problema que afeta o método de Roe). Ao mesmo tempo, por usar uma dissipação estritamente local, ele reduz drasticamente o borramento da solução. 

---

### **Extra: Fluxos HLL e HLLC**

Comentamos que o método de Roe atua de forma extremamente precisa para capturar choques, mas traz um alto custo computacional (exige autovetores e médias complexas) e pode sofrer com violações de entropia (choques de expansão). Por outro lado, o Rusanov (Lax-Friedrichs Local) é extremamente robusto e simples, mas sua dissipação simétrica acaba borrando detalhes importantes da solução.

Nesse cenário, na década de 1980 e 1990, pesquisadores buscaram uma alternativa que unisse o melhor dos dois mundos: a robustez contra violações de entropia e a simplicidade de cálculo do Lax-Friedrichs, mas com a inteligência direcional do Roe. Desse esforço nasceu a família de fluxos HLL.

#### **HLL (Harten, Lax e van Leer)**

Em 1983, Amiram Harten, Peter Lax e Bram van Leer propuseram uma mudança de paradigma. Em vez de tentar resolver toda a estrutura complexa de ondas de um Solucionador de Riemann exato (como o Roe faz usando a Jacobiana), por que não assumir uma estrutura simplificada na interface?

O HLL assume que a colisão de estados na interface produz apenas duas ondas principais:
* Uma onda mais rápida viajando para a esquerda, com velocidade $S_L$.
* Uma onda mais rápida viajando para a direita, com velocidade $S_R$.

Essas duas ondas dividem o espaço-tempo na interface em três regiões distintas:
1. Região à esquerda da onda $S_L$: A perturbação ainda não chegou aqui, então o estado continua sendo o do elemento à esquerda ($U_L$) intacto.
2. Região à direita da onda $S_R$: A perturbação ainda não chegou aqui, então o estado continua sendo o do elemento à direita ($U_R$) intacto.
3. Região intermediária (entre $S_L$ e $S_R$): Região onde as ondas colidiram, gerando um único estado constante médio que chamamos de $U^{HLL}$.

#### A Equação do Fluxo HLL

Para determinar o fluxo na exata interface do elemento, o HLL avalia três cenários com base nas velocidades das ondas estimadas ($S_L$ e $S_R$):
* Caso 1: Escoamento supersônico para a direita ($0 \leq S_L$)  
Ambas as ondas viajam para a direita. A interface é banhada apenas pela informação da esquerda:
$$F_{interface} = F_L$$
* Caso 2: Escoamento supersônico para a esquerda ($S_R \leq 0$)  
Ambas as ondas viajam para a esquerda. A interface é banhada apenas pela informação da direita:
$$F_{interface} = F_R$$
* Caso 3: Região subsônica ($S_L < 0 < S_R$)  
A interface está presa entre as duas ondas. Aplicando a lei de conservação integral num volume de controle entre $S_L$ e $S_R$, obtemos a equação central do fluxo HLL:
$$F^{HLL} = \frac{S_R F_L - S_L F_R + S_L S_R (U_R - U_L)}{S_R - S_L}$$

#### Conexão com o Lax-Friedrichs

Para entender por que o HLL é considerado um Lax-Friedrichs "esperto", podemos fazer um exercício matemático.

Imagine que simplificamos o problema e dizemos que a onda da direita e a da esquerda possuem a mesma velocidade máxima em módulos opostos ($S_R = C$ e $S_L = -C$). Substituindo isso na fórmula central do HLL:
$$F^{HLL} = \frac{C F_L - (-C) F_R + (-C) C (U_R - U_L)}{C - (-C)}$$

Simplificando os termos e dividindo por $2C$, a equação colapsa exatamente em:
$$F^{HLL} = \frac{1}{2}(F_L + F_R) - \frac{1}{2}C(U_R - U_L)$$

Esta é exatamente a equação clássica do Lax-Friedrichs! Enquanto o LxF chuta uma velocidade global simétrica $C$, o HLL calcula velocidades direcionais $S_L$ e $S_R$, injetando apenas a dissipação estritamente necessária. Isso o torna imune à violação de entropia do Roe.

#### O Defeito do HLL  
Para as equações de Euler, a física real possui três ondas, e não duas. A onda intermediária — que viaja na velocidade do fluido ($u$) — representa a descontinuidade de contato. Como o HLL original ignora essa onda central e assume um único estado constante na região intermediária, ele acaba gerando muita dissipação numérica, "borrando" as interfaces de contato e as camadas de cisalhamento.

#### **HLLC (Toro, Spruce e Speares - 1994)**
Para sanar o principal defeito do HLL sem perder sua robustez, o Prof. Eleuterio Toro e seus colaboradores introduziram o HLLC (onde a letra C refere-se a Contact / Contato).

A ideia fundamental foi manter a simplicidade e a robustez das velocidades externas ($S_L$ e $S_R$), mas restaurar explicitamente a onda central ($S_*$), que representa o transporte convectivo e a descontinuidade de contato do fluido. Isso divide a região intermediária em duas sub-regiões.

O resultado: O HLLC uniu o melhor de todos os mundos. Ele entrega uma precisão comparável à do método de Roe para capturar choques e interfaces de contato de forma extremamente nítida, mas mantém a garantia matemática e a robustez do HLL de nunca gerar soluções termodinâmicas impossíveis, exigindo um esforço computacional bem menor que o Roe.

> Nota Prática: Se o escoamento for inteiramente supersônico (com todas as ondas indo para a mesma direção, $S_L > 0$), o Roe, o HLL e o HLLC percebem essa condição de contorno e convergem matematicamente para o esquema Upwind clássico.

Para modelar escoamentos compressíveis avançados, problemas aeronáuticos e aeroespaciais com fortes choques, o HLLC consolidou-se como uma das escolhas mais populares e eficientes na comunidade de CFD.

---

## **Fluxos Numéricos Difusivos**

Na **Aula 06**, comentamos sobre problemas que contêm difusão e como eles alteram o formato da matriz de rigidez e os operadores locais. Para os fluxos numéricos nas fronteiras, o tratamento também precisa ser diferente.

Nesta seção, iremos focar nos principais fluxos numéricos que aparecem na solução de problemas difusivos na literatura de Galerkin Descontínuo:
* Fluxo Central (Central Flux)
* Local Galerkin Discontinuos (LDG)
* Interior penalty flux (IP)
* Bassi - Rebay 1 e 2 (BR1 e BR2)

### **Problemas Difusivos e a Formulação Fraca**

Assim como vimos na Aula 06, os fluxos numéricos difusivos exigem que reescrevamos a equação de segunda ordem como um sistema de primeira ordem. Podemos aplicar esse formalismo partindo de um problema puramente difusivo ou considerando uma viscosidade artificial. Tomaremos como exemplo a equação mais simples, a Equação de Poisson:

$$-\nabla^2 u(x) = f(x)$$

como fizemos na aula 06 podemos separar tal equação em duas:
$$-\nabla \cdot q = f \\ q = \nabla u$$

Agora, multiplicamos ambas as equações por funções de teste (digamos, $w$ para a primeira e $v$ para a segunda) e integramos sobre o volume de um elemento $\Omega_e$:

$$\int_{\Omega_e} q \cdot w \, dx = \int_{\Omega_e} \nabla u \cdot w \, dx$$

$$\int_{\Omega_e} f v \, dx = - \int_{\Omega_e} (\nabla \cdot q) v \, dx $$

Aplicando a integração por partes (Teorema da Divergência) no termo gradiente da primeira equação e no termo divergente da segunda, fazemos surgir as integrais de contorno. É exatamente aqui que os fluxos numéricos entram em cena:

$$\int_{\Omega_e} q \cdot w \, dx = - \int_{\Omega_e} u (\nabla \cdot w) \, dx + \int_{\partial \Omega_e} \mathbf{u^*} (w \cdot n) \, ds$$

$$\int_{\Omega_e} f v \, dx = \int_{\Omega_e} q \cdot \nabla v \, dx - \int_{\partial \Omega_e} (\mathbf{q^*} \cdot n) v \, ds $$



#### A Formulação Matricial do Problema Difusivo

Para conectarmos essa teoria matemática diretamente à forma como o computador resolve o problema, é fundamental escrevermos essas integrais no seu formato matricial discreto.Lembrando que, em 1D, o vetor normal $n$ aponta para fora do elemento (assumindo o valor $+1$ na fronteira direita e $-1$ na fronteira esquerda), a primeira equação (que calcula a variável auxiliar $q$) assume a seguinte forma matricial:

$$\underbrace{J_e \left[\int_{\Omega_{pd}} \phi_i \phi_j d \xi \right]}_{\text{Matriz de Massa} \ \mathcal{M}} \begin{Bmatrix} q_0 \\ q_1 \\ \vdots \\ q_P \end{Bmatrix} =  - \underbrace{\int_{\Omega_{pd}} u \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi}_{\text{Transposta da Rigidez} \ \mathcal{S}^T \ u} +  \underbrace{u^*_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} -  u^*_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}}_{\text{Fluxo Numérico (Traços de } u^* \text{)}}$$

Assim como fizemos para o operador convectivo, podemos isolar o vetor de coeficientes de $q$, multiplicando toda a equação pela inversa da Matriz de Massa ($\mathcal{M}^{-1}$):

$$\begin{Bmatrix} q_0 \\ q_1 \\ \vdots \\ q_P \end{Bmatrix} = \frac{1}{J_e} \left[\int_{\Omega_{pd}} \phi_i \phi_j d \xi \right]^{-1} \left(  - \int_{\Omega_{pd}} u \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi +  u^*_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} -  u^*_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix} \right)$$

De forma análoga, a segunda equação (que avalia o balanço da física principal do elemento) é expandida para:

$$\underbrace{J_e \left[ \int_{\Omega_{pd}} f \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi \right]}_{\text{Vetor de Força } F} = \underbrace{\int_{\Omega_{pd}} q \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi}_{\text{Matriz de Rigidez} \ \mathcal{S} \ q} -  \underbrace{\left( q^*_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} -  q^*_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix} \right)}_{\text{Fluxo Numérico (Traços de } q^* \text{)}}$$

Observe o surgimento de $u^*$ e $q^*$ nos termos de fronteira. Como a malha é descontínua, os valores nas interfaces não são únicos (temos o estado interno e o estado do vizinho). Para acoplar os elementos, precisamos substituir esses traços por formulações de fluxos numéricos.

### **Construindo os Fluxos Numéricos Difusivos**

Para definir $u^*$ e $q^*$, a literatura utiliza amplamente os operadores de média $\{\{ \cdot \}\}$ e salto $[\![ \cdot ]\!]$ avaliados na interface.

A tabela abaixo resume como as principais estratégias difusivas combinam esses operadores para construir os fluxos numéricos:

<div align="center">

| Fluxo Numérico | $u^*$ | $q^*$ |
| :------------: |:------:|:-----:|
|Fluxo Central | $\{\{ u_h \}\}$ | $\{\{ q_h \}\} - \tau [\![ u_h ]\!]$|
|Local DG | $\{\{ u_h \}\} + \beta \cdot [\![ u_h ]\!]$ | $\{\{ q_h \}\} - \beta[\![ q_h ]\!]  - \tau[\![ u_h ]\!]$|
|IP       | $\{\{ u_h \}\}$| $\{\{\nabla u_h \}\} - \tau[\![ u_h ]\!]$ |
|BR1      | $\{\{ u_h \}\}$ | $\{\{ q_h \}\}$ |
|BR2      | $\{\{ u_h \}\}$ | $\{\{ q_h \}\} - \eta \{ r_f([\![ u_h ]\!]) \}$|

</div>

Entendendo os termos da Tabela:
* $\{\{ u_h \}\}$ e $\{\{ q_h \}\}$ (Média): Representam a média aritmética entre o valor do elemento e do seu vizinho.  
$$\{\{ u_h \}\} = \dfrac{u^- + u^+}{2}$$
* $[\![ u_h ]\!]$ e $[\![ q_h ]\!]$ (Salto): Medem a diferença (a descontinuidade) entre os estados na interface.
$$[\![ u_h ]\!] = \hat{n}^- u^- + \hat{n}^+ u^+$$
* $\tau$ (Penalização): É um parâmetro de estabilização. Ele pune saltos muito grandes de $u_h$, forçando a solução a ser razoavelmente contínua.
* $\beta$ (Vetor direcional): Utilizado no método LDG para alternar a direção da informação (se a informação de $u$ vem mais da esquerda, a de $q$ virá mais da direita).
* $r_f([\![ u_h ]\!])$ (Levantamento Local): Utilizado no método BR2. É um operador matricial que pega o salto da fronteira e o converte (levanta) em uma correção de fluxo local, e $\eta$ é uma constante de estabilidade (geralmente relacionada ao número de faces do elemento).

### **Breve conexão com a Implementação**

Como já sabemos, nas nossas implementações não calculamos as integrais de fronteira diretamente a cada iteração. Em vez disso, utilizamos as Matrizes de Levantamento (Lift Matrices) (que vimos na **aula 06**). Em 1D, costumamos ter:
 
* $\mathcal{F}_{L,k}$ (Flk): Matriz da face esquerda (Left) do elemento atual $k$.
* $\mathcal{F}_{R,k}$ (Frk): Matriz da face direita (Right) do elemento atual $k$.
* $\mathcal{F}_{L,k+1}$ (Flkp1): Matriz da face esquerda do elemento vizinho à direita ($k+1$).
* $\mathcal{F}_{R,k-1}$ (Frkm1): Matriz da face direita do elemento vizinho à esquerda ($k-1$).

Vamos ver como as escolhas de fluxos numéricos da nossa tabela teórica ditam quais dessas matrizes serão usadas em cada método.

#### O Método LDG (Local Discontinuous Galerkin)

NNa formulação teórica do LDG, escolhemos alternar a direção da informação através do vetor direcional $\beta$. Em 1D, isso frequentemente se traduz em pegar a informação do fluxo puramente de um lado (100% upwind ou 100% downwind) para calcular a variável auxiliar $q$.

Se olharmos para o cálculo de $q$ na solução do problema de Allen-Cahn, o algoritmo adota $u^* = u^+$ (pega apenas a informação que vem do vizinho). Matematicamente, a forma fraca discreta fica:

$$q_k = \mathcal{M}^{-1} J^{-1} \Big( -\mathcal{S}^T u_k + \mathcal{F}_{L,k+1} u_{k+1} - \mathcal{F}_{L,k} u_k \Big)$$

Observe como essa equação se traduz de forma idêntica e sem tirar nem pôr no código:
```Python
### Dissipative operator discretized with LDG scheme
qt[:,1:-1] = (InvM*( J[:]**(-1)*(- np.dot(S.T,ut[:,1:-1,0]) \
                                + np.dot(Flkp1,ut[:,2:,0]) \
                                - np.dot(Flk,ut[:,1:-1,0]) ) ).T).T.copy()
```

Repare que não há a presença do fator $0.5$ (que indicaria uma média). O código busca o estado do vizinho da direita (`Flkp1, ut[:,2:,0]`) e avalia o elemento interno na sua face esquerda (`Flk, ut[:,1:-1,0]`), garantindo a comunicação direcional característica do LDG.

### **Como escolher o Fluxo Numérico correto?**

A escolha do Fluxo Numérico depende diretamente da **natureza física da equação** que estamos resolvendo.

- **Problemas Convectivos:** a informação propaga-se em direções bem definidas, como acontece na equação da advecção, em escoamentos invíscidos ou no tráfego de veículos. Nesses casos, o Fluxo Numérico precisa identificar corretamente a direção de propagação das ondas.

- **Problemas Difusivos:** a informação espalha-se em todas as direções, como ocorre na condução de calor ou em fluidos viscosos. Nesses problemas, o Fluxo Numérico deve garantir estabilidade e consistência na aproximação das derivadas espaciais.

Ao longo dos anos, diversas estratégias foram propostas para tratar cada uma dessas situações. A tabela abaixo apresenta alguns dos Fluxos Numéricos mais importantes e que serão estudados ao longo do curso.


| Nome do Fluxo Numérico | Natureza Física | Ideia Principal |
|:---|:---:|:---|
| Upwind (Godunov) | Convectivo | Utiliza apenas a informação proveniente da direção de propagação da onda. |
| Roe | Convectivo | Lineariza localmente o problema utilizando uma Jacobiana aproximada na interface. |
| Lax-Friedrichs (Rusanov) | Convectivo | Calcula uma média entre os estados vizinhos e adiciona dissipação numérica para estabilizar a solução. |
| HLL / HLLC | Convectivo | Estima apenas as ondas dominantes que atravessam a interface para construir o fluxo. |
| Interior Penalty (SIPG/NIPG) | Difusivo | Introduz um termo de penalização para manter a estabilidade da solução nas interfaces. |
| Local Discontinuous Galerkin (LDG) | Difusivo | Reescreve a equação de segunda ordem como um sistema de equações de primeira ordem. |
| Bassi-Rebay (BR1/BR2) | Difusivo | Utiliza operadores de *lifting* para reconstruir gradientes nas interfaces antes de calcular o fluxo. |